# 의미 고정 N/V M2 + 격리 first-hop M3 + 양성가중 M4: Dunnhumby seed 42

역사적 개발구간(684~690일)에서 B′·실제 CLV M3·degree 내부 CLV 순열 M3·관계-only M3를 각각 100 epoch 공동학습합니다. M2와 M4는 모든 arm에서 동일하며 M3의 사용자 1차 전파 gate만 달라집니다. test와 holdout은 만들지 않습니다.

학습 전 q_C 공유, 순열 불변식, first-hop 강도 0.075, 사용자별 전파 총량 보존, B′ 계산 동일성 및 gradient 연결을 검사합니다. 중단 후 같은 셀을 다시 실행하면 완료 epoch부터 자동 재개합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'f4bc5b8166ec28b19c13008ad7a8c63b826876f8'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_semantic_nv_m3_screen import (
    ACTUAL_MODEL_ID,
    M3_OFF_MODEL_ID,
    RELATION_MODEL_ID,
    SHUFFLE_MODEL_ID,
    configure_semantic_nv_m3_screen,
    preflight_summary,
    run_semantic_nv_m3_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_semantic_nv_m3_screen(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_semantic_nv_isolated_m3_screen_v1',
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == [
    M3_OFF_MODEL_ID, ACTUAL_MODEL_ID, SHUFFLE_MODEL_ID, RELATION_MODEL_ID
]
assert summary['m2_frozen_definition']['rho'] == 0.15
assert summary['m2_frozen_definition']['beta'] == 0.25
assert summary['m3']['target_log_coefficient_ratio_std'] == 0.075
assert summary['fixed']['test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_semantic_nv_m3_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) B′·actual·M3 순열·관계-only 절대지표')
show(result_df)
print('2) 대조군별 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) ID 점수 대비 경제점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) Top-10 변화')
show(result_df.attrs['top10_overlap'])
print('5) 정답 신규 진입·이탈')
show(result_df.attrs['truth_flow'])
print('6) 개입 작동 점검')
print(json.dumps(result_df.attrs['operational_checks'], ensure_ascii=False, indent=2))
print('7) 사전 고정 판독')
print(json.dumps(result_df.attrs['outcome_reading'], ensure_ascii=False, indent=2))
print('8) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))